# 🧪 Exhibition Connector RAG — Live API & Hugging Face Test

این نوت‌بوک سبک برای تست نسخه‌ی جدید پروژه است:

- Vercel API backend
- Hugging Face static frontend
- RAG/Search endpoint
- Scraper endpoint
- خروجی‌های نمونه برای شرکت‌ها

بدون نیاز به GPU، FAISS یا توکن Hugging Face اجرا می‌شود.


In [ ]:
# تنظیم لینک‌ها
VERCEL_API_BASE = "https://vercel-app-amber-five.vercel.app"
HF_FRONTEND_URL = "https://sosa123456-exhibition-connector-rag2-static.static.hf.space/index.html?v=20260726-10"

print("Vercel API:", VERCEL_API_BASE)
print("Hugging Face Frontend:", HF_FRONTEND_URL)


In [ ]:
import requests, json, textwrap, pandas as pd

def get_json(url, **params):
    r = requests.get(url, params=params, timeout=45)
    print("GET", r.url)
    print("status:", r.status_code)
    r.raise_for_status()
    return r.json()

def post_json(url, payload):
    r = requests.post(url, json=payload, timeout=45)
    print("POST", url)
    print("status:", r.status_code)
    r.raise_for_status()
    return r.json()


## 1) تست سلامت API

In [ ]:
health = get_json(f"{VERCEL_API_BASE}/api/health")
health


## 2) تست RAG/Search با چند سؤال

In [ ]:
queries = [
    "وب‌سایت شرکت پریسماتک چیست؟",
    "مواد شیمیایی تصفیه آب",
    "شرکت‌های مرتبط با ابزار دقیق کدامند؟",
    "سالن 31B",
]

rows = []
for q in queries:
    data = get_json(f"{VERCEL_API_BASE}/api/search", q=q, limit=5)
    for rank, item in enumerate(data.get("results", [])[:5], start=1):
        rows.append({
            "query": q,
            "rank": rank,
            "score": round(float(item.get("score", 0)), 2),
            "company": item.get("company"),
            "hall": item.get("hall") or "—",
            "booth": item.get("booth") or "—",
            "website": item.get("website") or "—",
        })

df = pd.DataFrame(rows)
df


## 3) تست دقیق پریسماتک باید رتبه اول باشد

In [ ]:
prisma = get_json(f"{VERCEL_API_BASE}/api/search", q="وب‌سایت شرکت پریسماتک چیست؟", limit=3)
first = prisma["results"][0]
print("Top result:", first["company"], first.get("website"))
assert "پریسماتک" in first["company"], "❌ پریسماتک رتبه اول نیست"
print("✅ تست پریسماتک موفق بود")


## 4) تست Scraper شرکت / URL

In [ ]:
scrape = get_json(f"{VERCEL_API_BASE}/api/scrape", url="https://prismatech.ir/")
print("title:", scrape["scraped"].get("title"))
print("emails:", scrape["scraped"].get("emails"))
print("phones:", scrape["scraped"].get("phones")[:3])
print("snippet:", scrape["scraped"].get("textSnippet", "")[:300])


## 5) تست search + scrape همزمان

In [ ]:
combo = get_json(f"{VERCEL_API_BASE}/api/search", q="پریسماتک", limit=3, scrape=1)
print("results:", [r["company"] for r in combo.get("results", [])])
print("scraped title:", combo.get("scrapedUpdate", {}).get("title"))


## 6) تست فرانت Hugging Face

In [ ]:
r = requests.get(HF_FRONTEND_URL, timeout=45)
print("status:", r.status_code)
print("bytes:", len(r.text))
assert r.status_code == 200
assert "Exhibition Connector" in r.text or "پرتال" in r.text
print("✅ فرانت Hugging Face در دسترس است")


## 7) خلاصه نتیجه تست

In [ ]:
summary = {
    "api_ok": health.get("ok"),
    "companies": health.get("stats", {}).get("companies"),
    "halls": health.get("stats", {}).get("halls"),
    "frontend_url": HF_FRONTEND_URL,
    "api_url": VERCEL_API_BASE,
}
summary
